# Ch.1 — Linear Regression

> Find the line that minimises the average squared error. Every neural network you will see later is just many of these stacked with non-linearities between them.

**Dataset:** California Housing — `sklearn.datasets.fetch_california_housing`  
**Task:** Predict median house value (`MedHouseVal`) from median income (`MedInc`)

## The Core Idea

Linear regression fits the function:

$$\hat{y} = \mathbf{W}^\top \mathbf{x} + b$$

- $\mathbf{W}$ — weights (one per feature): how much each feature moves the prediction  
- $b$ — bias: shifts the line up or down  
- $\hat{y}$ — predicted value  

Training = adjusting $\mathbf{W}$ and $b$ to minimise the Mean Squared Error (MSE) loss.

## Running Example

You are a data scientist at a real estate platform. First task: estimate median house value for a California district given its median income. One input, one output — the simplest model that contains all the ideas that scale to deep networks.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42

print("Libraries loaded.")

In [ ]:
# ── Load the California Housing dataset ───────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f"Rows: {len(df):,}   Columns: {df.shape[1]}")
print(f"\nFeatures: {list(housing.feature_names)}")
print(f"Target  : MedHouseVal (median house value in $100k units)")
df.head()

In [ ]:
# ── Single-feature setup: MedInc → MedHouseVal ────────────────────────────
X_single = df[["MedInc"]].values      # shape (n, 1)
y         = df["MedHouseVal"].values  # shape (n,)

X_train, X_test, y_train, y_test = train_test_split(
    X_single, y, test_size=0.2, random_state=SEED
)

# Scale — important for gradient descent; not needed for sklearn closed-form,
# but we scale here so the same pipeline works for both.
scaler  = StandardScaler()
Xtr_s   = scaler.fit_transform(X_train)
Xte_s   = scaler.transform(X_test)

print(f"Train: {Xtr_s.shape}  Test: {Xte_s.shape}")

## The Math

### Loss Functions

| Loss | Formula | Key property |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(\hat{y}_i - y_i)^2$ | Large errors penalised heavily; differentiable |
| **MAE** | $\frac{1}{n}\sum|\hat{y}_i - y_i|$ | Robust to outliers; not differentiable at 0 |
| **RMSE** | $\sqrt{\text{MSE}}$ | Same units as target; easier to interpret |

### Evaluation Metrics

$$R^2 = 1 - \frac{\sum(\hat{y}_i - y_i)^2}{\sum(\bar{y} - y_i)^2} \qquad \bar{R}^2 = 1 - (1 - R^2)\cdot\frac{n-1}{n-p-1}$$

$R^2 = 1$ → perfect. $R^2 = 0$ → no better than predicting the mean. Always use **Adjusted $R^2$** when comparing models with different numbers of features.

In [ ]:
# ── Implement loss functions from scratch ─────────────────────────────────
def mse(y_true, y_pred):  return np.mean((y_pred - y_true) ** 2)
def mae(y_true, y_pred):  return np.mean(np.abs(y_pred - y_true))
def rmse(y_true, y_pred): return np.sqrt(mse(y_true, y_pred))

def r2(y_true, y_pred):
    ss_res = np.sum((y_pred - y_true) ** 2)
    ss_tot = np.sum((y_true.mean() - y_true) ** 2)
    return 1 - ss_res / ss_tot

def adj_r2(y_true, y_pred, n_features):
    n = len(y_true)
    return 1 - (1 - r2(y_true, y_pred)) * (n - 1) / (n - n_features - 1)

# Quick sanity check on dummy data
y_dummy = np.array([1.0, 2.0, 3.0])
p_dummy = np.array([1.1, 1.9, 3.2])
print(f"MSE:  {mse(y_dummy, p_dummy):.4f}")
print(f"MAE:  {mae(y_dummy, p_dummy):.4f}")
print(f"RMSE: {rmse(y_dummy, p_dummy):.4f}")
print(f"R²:   {r2(y_dummy, p_dummy):.4f}")

## Step by Step — Manual Gradient Descent

The update rule for a single-weight model:

$$\frac{\partial \text{MSE}}{\partial W} = \frac{2}{n} \mathbf{X}^\top(\hat{y} - y) \qquad W \leftarrow W - \alpha \cdot \frac{\partial \text{MSE}}{\partial W}$$

Running this in a loop for many epochs is training.

In [ ]:
# ── Gradient descent from scratch ─────────────────────────────────────────
# Works for any number of features — X must be shape (n, p).
# In this chapter p=1 (MedInc only); later chapters pass all 8 features.
def gradient_descent(X, y, alpha=0.05, epochs=300):
    n, p = X.shape
    W = np.zeros(p)
    b = 0.0
    history = []

    for epoch in range(epochs):
        y_hat = X @ W + b
        error = y_hat - y

        dW = (2 / n) * X.T @ error
        db = (2 / n) * error.sum()

        W -= alpha * dW
        b -= alpha * db

        history.append(mse(y, y_hat))

    return W, b, history

W_gd, b_gd, loss_history = gradient_descent(Xtr_s, y_train, alpha=0.05, epochs=300)

y_pred_gd = Xte_s @ W_gd + b_gd

print(f"Gradient Descent → W: {W_gd[0]:.4f}  b: {b_gd:.4f}")
print(f"Test RMSE: {rmse(y_test, y_pred_gd):.4f}  R²: {r2(y_test, y_pred_gd):.4f}")

In [ ]:
# ── Plot: loss curve ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(loss_history, color="#2980b9", linewidth=2)
ax.set(title="Training Loss Curve (MSE) — Gradient Descent",
       xlabel="Epoch", ylabel="MSE")
ax.axhline(loss_history[-1], color="#e74c3c", linestyle="--",
           label=f"Final MSE: {loss_history[-1]:.4f}")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── sklearn closed-form solution for comparison ───────────────────────────
model = LinearRegression()
model.fit(Xtr_s, y_train)
y_pred_sk = model.predict(Xte_s)

print("sklearn LinearRegression (closed-form):")
print(f"  W: {model.coef_[0]:.4f}  b: {model.intercept_:.4f}")
print(f"  Test RMSE: {rmse(y_test, y_pred_sk):.4f}  R²: {r2(y_test, y_pred_sk):.4f}")

In [ ]:
# ── Plot: fitted line on scatter ───────────────────────────────────────────
# Unscale for readable axis labels
x_plot   = np.linspace(Xte_s.min(), Xte_s.max(), 200).reshape(-1, 1)
y_line   = model.predict(x_plot)
x_plot_u = scaler.inverse_transform(x_plot)  # back to original MedInc scale

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(scaler.inverse_transform(Xte_s), y_test,
           alpha=0.3, s=10, color="#2980b9", label="Test districts")
ax.plot(x_plot_u, y_line, color="#e74c3c", linewidth=2.5, label="Fitted line")
ax.set(title="MedInc vs MedHouseVal — Linear Regression Fit",
       xlabel="Median Income ($10k)", ylabel="Median House Value ($100k)")
ax.legend()
plt.tight_layout()
plt.show()

## The Hyperparameter Dial — Learning Rate

The learning rate $\alpha$ controls how large each gradient step is.

| $\alpha$ | Effect |
|---|---|
| Too high (e.g. 2.0) | Loss oscillates or diverges |
| Good (e.g. 0.05) | Loss decreases smoothly |
| Too low (e.g. 0.0001) | Loss barely moves per epoch |

In [ ]:
# ── Learning rate sweep ────────────────────────────────────────────────────
alphas = {"Too high (2.0)": 2.0, "Good (0.05)": 0.05, "Too low (0.0001)": 0.0001}
epochs = 100

fig, ax = plt.subplots(figsize=(9, 5))
colors  = ["#e74c3c", "#27ae60", "#2980b9"]

for (label, alpha), color in zip(alphas.items(), colors):
    _, _, history = gradient_descent(Xtr_s, y_train, alpha=alpha, epochs=epochs)
    # Clip extreme values for readability
    clipped = [min(h, 20) for h in history]
    ax.plot(clipped, label=label, color=color, linewidth=2)

ax.set(title="Learning Rate Effect on Loss Curve",
       xlabel="Epoch", ylabel="MSE (clipped at 20)")
ax.legend()
plt.tight_layout()
plt.show()

## What Can Go Wrong — Trap Demo

### Trap: R² always improves with more features

Adding a pure noise column makes training R² go up even though the model learned nothing real. Adjusted R² correctly penalises for the extra feature.

In [ ]:
# ── R² inflation trap ──────────────────────────────────────────────────────
rng = np.random.default_rng(SEED)

X_all = df[housing.feature_names].values
X_tr_all, X_te_all, y_tr, y_te = train_test_split(X_all, y, test_size=0.2, random_state=SEED)

# Baseline: 8 real features
sc2 = StandardScaler()
Xtr2 = sc2.fit_transform(X_tr_all)
Xte2 = sc2.transform(X_te_all)
m2   = LinearRegression().fit(Xtr2, y_tr)
p2   = m2.predict(Xte2)
r2_base = r2(y_tr, m2.predict(Xtr2))
ar2_base = adj_r2(y_tr, m2.predict(Xtr2), n_features=8)

# Add 5 pure noise columns
noise = rng.standard_normal((len(X_tr_all), 5))
Xtr3  = np.hstack([Xtr2, noise])
noise_te = rng.standard_normal((len(X_te_all), 5))
Xte3 = np.hstack([Xte2, noise_te])
m3   = LinearRegression().fit(Xtr3, y_tr)
r2_noisy  = r2(y_tr, m3.predict(Xtr3))
ar2_noisy = adj_r2(y_tr, m3.predict(Xtr3), n_features=13)

print("Model         |  Train R²  |  Adjusted R²")
print("-" * 44)
print(f"8 real feats  |  {r2_base:.4f}   |  {ar2_base:.4f}")
print(f"+ 5 noise     |  {r2_noisy:.4f}   |  {ar2_noisy:.4f}  ← penalised correctly")

In [ ]:
# ── Residual plot: does the linear assumption hold? ────────────────────────
residuals = y_test - y_pred_sk

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_pred_sk, residuals, alpha=0.3, s=8, color="#2980b9")
axes[0].axhline(0, color="#e74c3c", linewidth=1.5, linestyle="--")
axes[0].set(title="Residuals vs Fitted",
            xlabel="Fitted values", ylabel="Residuals")

axes[1].hist(residuals, bins=60, color="#2980b9", edgecolor="white")
axes[1].set(title="Residual Distribution",
            xlabel="Residual", ylabel="Count")

plt.suptitle("Residual Diagnostics — MedInc → MedHouseVal", y=1.02)
plt.tight_layout()
plt.show()
print("Note: the funnel shape (heteroscedasticity) suggests the linear model\n"
      "has more error in high-value districts — a non-linear model would do better.")

## Exercises

Work through these before moving to Ch.2. Each one targets a specific concept from the chapter.

**Exercise 1 — Multi-feature model**  
Refit the model using all 8 housing features instead of just `MedInc`. Compare RMSE and R² against the single-feature baseline. Does adding features help?

**Exercise 2 — Learning rate grid**  
Run gradient descent with α ∈ `[0.001, 0.01, 0.1, 0.5, 1.0]` for 200 epochs each. Plot all loss curves on one chart. At what value does it first diverge?

**Exercise 3 — Outlier robustness**  
There are districts with `MedHouseVal = 5.0` (capped at $500k). These act as outliers that inflate MSE. Use `sklearn.linear_model.SGDRegressor` to train two models: one with `loss='squared_error'` (MSE) and one with `loss='huber'` (a smooth approximation to MAE that is robust to outliers). Compare their test RMSE — Huber will hurt less on the capped values.

> Note: `SGDRegressor` does not support `loss='absolute_error'` directly. Huber loss is the practical stand-in — it behaves like MAE for large errors and like MSE for small ones.

In [ ]:
# ── Exercise 1 scaffold — multi-feature model ──────────────────────────────
# TODO: replace X_single with all 8 features and rerun the pipeline

X_multi = df[housing.feature_names].values   # shape (n, 8)

# YOUR CODE: split, scale, fit, evaluate
# ...

# Expected output: RMSE around 0.72, R² around 0.60 (vs ~0.48 for single feature)

In [ ]:
# ── Exercise 2 scaffold — learning rate grid ───────────────────────────────
alpha_grid = [0.001, 0.01, 0.1, 0.5, 1.0]

fig, ax = plt.subplots(figsize=(10, 5))

for alpha in alpha_grid:
    _, _, hist = gradient_descent(Xtr_s, y_train, alpha=alpha, epochs=200)
    # YOUR CODE: clip and plot each curve with a descriptive label
    pass

ax.set(title="Learning Rate Grid", xlabel="Epoch", ylabel="MSE")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Exercise 3 scaffold — outlier robustness: MSE vs Huber ───────────────
from sklearn.linear_model import SGDRegressor

# Districts where the value was capped at $500k: MedHouseVal == 5.0
capped_mask = y == 5.0
print(f"Capped districts: {capped_mask.sum()} ({100*capped_mask.mean():.1f}%)")

# YOUR CODE:
# mse_model  = SGDRegressor(loss='squared_error', max_iter=1000, random_state=SEED)
# huber_model = SGDRegressor(loss='huber', max_iter=1000, random_state=SEED)

# Fit both on (Xtr_s, y_train), evaluate on (Xte_s, y_test)# ...
# Print RMSE for each — Huber should be lower because the capped values hurt it less

---

## §3.5 Interactive Widget — Can You Eyeball the Best Fit?

Use the sliders to move the regression line by hand. Watch the MSE change in the title and the position of the loss-parabola cursor update in real time.

**Controls:**
- `w` — slope (weight on MedInc)
- `b` — intercept (bias)
- **"Run gradient descent from here"** — animates the GD path from your slider position to the optimum

> **Note:** This widget requires `ipywidgets`. Run this cell interactively in Jupyter or VS Code notebooks. Static HTML renders a screenshot only.

In [ ]:
"""
§3.5 Interactive eyeball widget — requires ipywidgets.
Interactive widget — run this cell in Jupyter/VS Code notebook.
Static HTML renders a screenshot only.
"""
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler

try:
    import ipywidgets as widgets
    from IPython.display import display
    _HAS_WIDGETS = True
except ImportError:
    _HAS_WIDGETS = False
    print("ipywidgets not installed. Run: pip install ipywidgets")

# ── Load and scale data ────────────────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
X_raw = housing.data["MedInc"].values
y_raw = housing.target.values

# Use a fixed subsample for speed in interactive mode
rng = np.random.default_rng(42)
idx = rng.choice(len(X_raw), size=500, replace=False)
X_sub, y_sub = X_raw[idx], y_raw[idx]

# ── Optimal OLS solution (reference) ──────────────────────────────────────
w_opt = np.cov(X_sub, y_sub)[0, 1] / np.var(X_sub)
b_opt = np.mean(y_sub) - w_opt * np.mean(X_sub)

# ── MSE parabola over a w-grid (b fixed at optimum for visualisation) ─────
w_grid = np.linspace(-0.5, 1.5, 300)
mse_grid = np.array([np.mean((w * X_sub + b_opt - y_sub) ** 2) for w in w_grid])


def _mse(w, b):
    return np.mean((w * X_sub + b - y_sub) ** 2)


def _gd_path(w0, b0, lr=0.05, epochs=120):
    """Run gradient descent from (w0, b0) and return trajectory."""
    w, b = w0, b0
    ws, bs = [w], [b]
    N = len(X_sub)
    for _ in range(epochs):
        err = w * X_sub + b - y_sub
        grad_w = (2 / N) * (err @ X_sub)
        grad_b = (2 / N) * err.sum()
        w -= lr * grad_w
        b -= lr * grad_b
        ws.append(w)
        bs.append(b)
        if abs(grad_w) < 1e-6 and abs(grad_b) < 1e-6:
            break
    return np.array(ws), np.array(bs)


def _draw(w, b, gd_ws=None, gd_bs=None):
    fig = plt.figure(figsize=(12, 5))
    gs = gridspec.GridSpec(1, 2, figure=fig)

    # Left: scatter + current line
    ax1 = fig.add_subplot(gs[0])
    ax1.scatter(X_sub, y_sub, alpha=0.15, s=10, color="#93c5fd", label="data")
    x_line = np.array([X_sub.min(), X_sub.max()])
    ax1.plot(x_line, w * x_line + b, color="#ef4444", lw=2, label=f"ŷ = {w:.3f}·x + {b:.3f}")
    ax1.plot(x_line, w_opt * x_line + b_opt, color="#22c55e", lw=1.5,
             linestyle="--", label=f"optimal (w={w_opt:.3f})")
    ax1.set_xlabel("MedInc (×$10k)")
    ax1.set_ylabel("MedHouseVal (×$100k)")
    ax1.set_title(f"MSE = {_mse(w, b):.4f}")
    ax1.legend(fontsize=8)

    # Right: MSE parabola + cursor + optional GD path
    ax2 = fig.add_subplot(gs[1])
    ax2.plot(w_grid, mse_grid, color="#6366f1", lw=2, label="MSE(w)  [b fixed at optimal]")
    ax2.scatter([w], [_mse(w, b_opt)], color="#ef4444", s=80, zorder=5, label="current w")
    ax2.axvline(w_opt, color="#22c55e", linestyle="--", lw=1, label=f"w* = {w_opt:.3f}")
    if gd_ws is not None:
        gd_mse = [_mse(gw, gb) for gw, gb in zip(gd_ws, gd_bs)]
        ax2.plot(gd_ws, gd_mse, color="#f97316", lw=1.5, marker=".", markersize=4,
                 alpha=0.8, label="GD path")
    ax2.set_xlabel("w  (slope)")
    ax2.set_ylabel("MSE")
    ax2.set_title("Loss parabola")
    ax2.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


if _HAS_WIDGETS:
    w_slider = widgets.FloatSlider(value=0.0, min=-0.5, max=1.5, step=0.01,
                                   description="w (slope):", continuous_update=True,
                                   style={"description_width": "initial"})
    b_slider = widgets.FloatSlider(value=0.0, min=-2.0, max=4.0, step=0.05,
                                   description="b (intercept):", continuous_update=True,
                                   style={"description_width": "initial"})
    run_btn = widgets.Button(description="▶ Run gradient descent from here",
                             button_style="info", layout=widgets.Layout(width="300px"))
    _gd_result = {"ws": None, "bs": None}

    def _on_run(_):
        ws, bs = _gd_path(w_slider.value, b_slider.value)
        _gd_result["ws"] = ws
        _gd_result["bs"] = bs
        _update(w_slider.value, b_slider.value)

    def _update(w, b):
        with out:
            out.clear_output(wait=True)
            _draw(w, b, _gd_result["ws"], _gd_result["bs"])

    run_btn.on_click(_on_run)
    out = widgets.Output()

    ui = widgets.VBox([
        widgets.HBox([w_slider, b_slider]),
        run_btn,
        out,
    ])

    widgets.interactive_output(_update, {"w": w_slider, "b": b_slider})
    display(ui)
    _update(w_slider.value, b_slider.value)
else:
    # Fallback static plot for non-interactive environments
    _draw(0.0, 0.0)